# Experiment: Oracle Beta vs Baselines

Objective:
- Reuse the saved oracle-beta diagnostics from `mcmcis_beta_diagnostics.ipynb`.
- Compare `oracle beta` against the pilot/simple beta initialization.
- Add `samc` and `iid` baselines at the same evaluation checkpoints.

Important interpretation note: the oracle-beta curves exclude the Optuna/HPO search cost. Treat `oracle beta` as an upper-bound reference, not as a fair end-to-end method budget.


In [ ]:
from __future__ import annotations

import concurrent.futures as cf
import json
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "perm_pval").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing perm_pval/ and results/.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.environ.setdefault("MPLCONFIGDIR", str(project_root / ".matplotlib"))

from perm_pval.experiments.notebook_studies import (
    SAMCWorkflowConfig,
    _effective_n_jobs,
    _iid_replicate_worker,
    _samc_replicate_worker,
    _try_make_process_pool,
    load_selected_scenarios,
    summarize_records,
    tune_samc_setup,
)

pd.set_option("display.max_columns", 120)
project_root


## Configuration

`BETA_RUN_DIR` points to the saved oracle-beta run. Set `SCENARIO_KEYS_OVERRIDE` to a list if you only want one or two scenarios; leave it as `None` to use all saved scenario folders in that run.


In [ ]:
CATALOG_PATH = project_root / "results" / "exact_scenarios" / "v1" / "catalog.json"
BETA_RUN_DIR = project_root / "results" / "mcmcis_beta_notebook" / "20260411_195452_beta_diag"
COMPARISON_DIR = BETA_RUN_DIR / "_oracle_vs_baselines"

# None means: use every scenario with saved oracle-beta outputs in BETA_RUN_DIR.
# Example for a quick first run: ["gwas_additive_score_sig_n60"]
SCENARIO_KEYS_OVERRIDE = None

RUN_BASELINES = True
SAVE_COMPARISON = True
BASELINE_BASE_SEED = 8_765_432
BASELINE_N_JOBS = min(6, os.cpu_count() or 1)
CONFIDENCE_LEVEL = 0.95

samc_cfg = SAMCWorkflowConfig(
    n_bins=100,
    t0=1_000.0,
    trace_every=200,
    lambda_min_pilot=10_000,
    proposal_size=2,
)

print(json.dumps({
    "BETA_RUN_DIR": str(BETA_RUN_DIR),
    "COMPARISON_DIR": str(COMPARISON_DIR),
    "SCENARIO_KEYS_OVERRIDE": SCENARIO_KEYS_OVERRIDE,
    "RUN_BASELINES": RUN_BASELINES,
    "BASELINE_N_JOBS": BASELINE_N_JOBS,
    "SAMC_PROPOSAL_SIZE": samc_cfg.proposal_size,
}, indent=2))


## Load Saved Oracle-Beta Results

This loads the `init` and `oracle` MCMC-IS records from the saved beta diagnostics run. Those records were produced by `run_named_mcmc_checkpoint_study` in `mcmcis_beta_diagnostics.ipynb`.


In [ ]:
def saved_scenario_keys(beta_run_dir: Path) -> list[str]:
    keys = []
    for child in sorted(beta_run_dir.iterdir()):
        if child.is_dir() and (child / "final_eval_records.jsonl").exists():
            keys.append(child.name)
    return keys


scenario_keys = list(SCENARIO_KEYS_OVERRIDE or saved_scenario_keys(BETA_RUN_DIR))
if not scenario_keys:
    raise RuntimeError(f"No saved oracle-beta scenario folders found under {BETA_RUN_DIR}")

scenarios = load_selected_scenarios(
    catalog_path=CATALOG_PATH,
    scenario_keys=scenario_keys,
    portfolio_group=None,
    min_tail_states=2,
)
scenario_by_key = {scenario.key: scenario for scenario in scenarios}

oracle_records = []
oracle_metadata = {}
for scenario_key in scenario_keys:
    scenario_dir = BETA_RUN_DIR / scenario_key
    records_df = pd.read_json(scenario_dir / "final_eval_records.jsonl", lines=True)
    metadata = json.loads((scenario_dir / "metadata.json").read_text())
    best_config = json.loads((scenario_dir / "best_config.json").read_text())
    oracle_metadata[scenario_key] = {"metadata": metadata, "best_config": best_config}
    for row in records_df.to_dict("records"):
        label = str(row.get("label"))
        row["scenario"] = scenario_key
        row["label"] = "simple init" if label == "init" else "oracle beta"
        row["method"] = row["label"]
        row["source"] = "saved_oracle_beta_run"
        oracle_records.append(row)

display(pd.DataFrame([
    {
        "scenario": key,
        "exact_p": scenario_by_key[key].exact_p,
        "checkpoints": tuple(oracle_metadata[key]["metadata"]["final_eval_settings"]["estimation_points"]),
        "repeats": int(oracle_metadata[key]["metadata"]["final_eval_settings"]["repeats"]),
        "simple_beta": oracle_metadata[key]["best_config"]["beta_init"],
        "oracle_beta": oracle_metadata[key]["best_config"]["best_beta"],
        "oracle_proposal_size": oracle_metadata[key]["best_config"]["best_proposal_size"],
        "oracle_selection_source": oracle_metadata[key]["best_config"].get("selection_source", "optuna"),
    }
    for key in scenario_keys
]))

print(f"Loaded {len(oracle_records):,} saved MCMC-IS records.")


## Run IID and SAMC Baselines

This runs only the missing non-MCMC baselines. Checkpoints and repeat counts are copied from the saved oracle-beta final evaluation settings for each scenario.


In [ ]:
def run_iid_samc_baselines_for_scenario(scenario_key: str) -> list[dict]:
    scenario = scenario_by_key[scenario_key]
    settings = oracle_metadata[scenario_key]["metadata"]["final_eval_settings"]
    checkpoints = tuple(int(v) for v in settings["estimation_points"])
    repeats = int(settings["repeats"])
    scenario_offset = 1_000_000 * scenario_keys.index(scenario_key)
    samc_setup = tune_samc_setup(
        scenario.problem,
        samc_cfg,
        seed=BASELINE_BASE_SEED + scenario_offset + 20_000,
    )

    jobs = []
    for rep in range(repeats):
        rep_seed = BASELINE_BASE_SEED + scenario_offset + 1_000 * rep
        jobs.append((
            "iid",
            {
                "scenario_key": scenario.key,
                "scenario_display": scenario.description,
                "problem": scenario.problem,
                "exact_p": float(scenario.exact_p),
                "checkpoints": checkpoints,
                "rep": rep,
                "rep_seed": rep_seed,
                "confidence_level": CONFIDENCE_LEVEL,
            },
        ))
        jobs.append((
            "samc",
            {
                "scenario_key": scenario.key,
                "scenario_display": scenario.description,
                "problem": scenario.problem,
                "exact_p": float(scenario.exact_p),
                "checkpoints": checkpoints,
                "samc_setup": samc_setup,
                "samc_cfg": samc_cfg,
                "rep": rep,
                "rep_seed": rep_seed,
            },
        ))

    def annotate_rows(kind: str, rows: list[dict]) -> list[dict]:
        for row in rows:
            row["label"] = kind
            row["method"] = kind
            row["source"] = "fresh_baseline_run"
        return rows

    rows = []
    n_workers = _effective_n_jobs(BASELINE_N_JOBS, len(jobs))
    executor = _try_make_process_pool(n_workers) if n_workers > 1 else None
    t0 = time.perf_counter()
    print(f"Running {scenario_key}: {len(jobs)} IID/SAMC jobs with n_workers={n_workers}")
    if executor is None:
        for kind, kwargs in jobs:
            worker = _iid_replicate_worker if kind == "iid" else _samc_replicate_worker
            rows.extend(annotate_rows(kind, worker(**kwargs)))
    else:
        with executor:
            future_to_kind = {}
            for kind, kwargs in jobs:
                worker = _iid_replicate_worker if kind == "iid" else _samc_replicate_worker
                future_to_kind[executor.submit(worker, **kwargs)] = kind
            for idx, future in enumerate(cf.as_completed(future_to_kind), start=1):
                kind = future_to_kind[future]
                rows.extend(annotate_rows(kind, future.result()))
                if idx % max(1, len(futures) // 5) == 0 or idx == len(futures):
                    print(f"  {scenario_key}: completed {idx}/{len(futures)} jobs")
    print(f"Finished {scenario_key} baselines in {time.perf_counter() - t0:.1f}s")
    return rows


if RUN_BASELINES:
    baseline_records = []
    for scenario_key in scenario_keys:
        baseline_records.extend(run_iid_samc_baselines_for_scenario(scenario_key))
else:
    baseline_path = COMPARISON_DIR / "baseline_records.jsonl"
    baseline_records = pd.read_json(baseline_path, lines=True).to_dict("records")

print(f"Baseline records: {len(baseline_records):,}")


## Combine and Summarize

The combined table has four labels: `oracle beta`, `simple init`, `samc`, and `iid`.


In [ ]:
combined_records = list(oracle_records) + list(baseline_records)
combined_summary = []
for scenario_key in scenario_keys:
    rows = [row for row in combined_records if row.get("scenario") == scenario_key]
    summary_rows = summarize_records(rows, group_fields=("checkpoint", "label"))
    for row in summary_rows:
        row["scenario"] = scenario_key
        row["exact_p"] = float(scenario_by_key[scenario_key].exact_p)
        combined_summary.append(row)

combined_records_df = pd.DataFrame(combined_records)
combined_summary_df = pd.DataFrame(combined_summary).sort_values(["scenario", "checkpoint", "label"])

method_order = ["oracle beta", "simple init", "samc", "iid"]
combined_summary_df["label"] = pd.Categorical(combined_summary_df["label"], categories=method_order, ordered=True)
combined_summary_df = combined_summary_df.sort_values(["scenario", "checkpoint", "label"])

if SAVE_COMPARISON:
    COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
    combined_records_df.to_json(COMPARISON_DIR / "combined_records.jsonl", orient="records", lines=True)
    pd.DataFrame(baseline_records).to_json(COMPARISON_DIR / "baseline_records.jsonl", orient="records", lines=True)
    combined_summary_df.to_json(COMPARISON_DIR / "combined_summary.json", orient="records", indent=2)

display(combined_summary_df[[
    "scenario",
    "checkpoint",
    "label",
    "n_runs",
    "exact_p",
    "mean_estimate",
    "rmse",
    "mean_abs_log10_error",
    "mean_variance_estimate",
    "mean_q_tilt_tail_share",
    "mean_ess",
    "mean_acceptance_rate",
    "mean_zero_rate",
    "mean_samc_max_rel_freq_error",
]])


## Max-Budget Leaderboard

This is the quickest table for article decisions: one row per scenario and method at the largest checkpoint.


In [ ]:
max_rows = []
for scenario_key, sub in combined_summary_df.groupby("scenario", observed=False):
    max_checkpoint = int(pd.to_numeric(sub["checkpoint"]).max())
    max_rows.append(sub[pd.to_numeric(sub["checkpoint"]) == max_checkpoint])
max_budget_df = pd.concat(max_rows, ignore_index=True).sort_values(["scenario", "rmse"])

display(max_budget_df[[
    "scenario",
    "checkpoint",
    "label",
    "mean_estimate",
    "rmse",
    "mean_abs_log10_error",
    "mean_q_tilt_tail_share",
    "mean_ess",
    "mean_zero_rate",
    "mean_samc_max_rel_freq_error",
]])

display(max_budget_df.pivot(index="scenario", columns="label", values="rmse"))


## Plots

These are intentionally simple diagnostic plots: mean estimate and RMSE over checkpoints for each method.


In [ ]:
def plot_scenario_comparison(summary_df: pd.DataFrame, scenario_key: str, save_dir: Path | None = None) -> Path | None:
    sub = summary_df[summary_df["scenario"] == scenario_key].copy()
    if sub.empty:
        return None
    exact_p = float(sub["exact_p"].iloc[0])
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), constrained_layout=True)
    colors = {
        "oracle beta": "#1f77b4",
        "simple init": "#ff7f0e",
        "samc": "#2ca02c",
        "iid": "#7f7f7f",
    }
    for label in method_order:
        label_sub = sub[sub["label"].astype(str) == label].sort_values("checkpoint")
        if label_sub.empty:
            continue
        x = label_sub["checkpoint"].astype(float).to_numpy()
        axes[0].plot(x, label_sub["mean_estimate"].astype(float), marker="o", linewidth=2, label=label, color=colors.get(label))
        axes[1].plot(x, label_sub["rmse"].astype(float), marker="o", linewidth=2, label=label, color=colors.get(label))

    axes[0].axhline(exact_p, color="black", linestyle="--", linewidth=1.2, label="exact p")
    axes[0].set_ylabel("mean estimate")
    axes[1].set_ylabel("RMSE")
    for ax in axes:
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("checkpoint")
        ax.grid(axis="y", alpha=0.25)
        ax.spines[["top", "right"]].set_visible(False)
    axes[0].legend(fontsize=8)
    fig.suptitle(f"{scenario_key}: oracle beta vs baselines", fontsize=12)

    out_path = None
    if save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)
        out_path = save_dir / f"{scenario_key}_oracle_vs_baselines.png"
        fig.savefig(out_path, dpi=170, bbox_inches="tight")
    plt.show()
    return out_path


plot_paths = []
for scenario_key in scenario_keys:
    path = plot_scenario_comparison(combined_summary_df, scenario_key, COMPARISON_DIR if SAVE_COMPARISON else None)
    if path is not None:
        plot_paths.append(path)

plot_paths
